In [ ]:
import sys, subprocess
from pathlib import Path
from google.colab import drive

def section(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

section("SETUP: dictionary-senses experiment, step 1 (CPU, label-free)")
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
print(f"Project folder : {PROJECT}")
print(f"Project found  : {'PASS' if (PROJECT / 'src/semeval_task3/data.py').is_file() else 'FAIL: check the path'}")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "requests", "scikit-learn", "scipy", "bcubed==1.5"], check=True)
print("Packages installed: PASS\nSetup complete. Run the next cell.")

In [ ]:
import json, hashlib, time, requests
from collections import Counter, defaultdict
from datetime import datetime, timezone
from project_utils import new_run, save_json, record_environment
from semeval_task3.data import load_dataset

def section(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

RUN = new_run(PROJECT, "step06_dictionary_senses")
LANG_NAME = {"SV": "Swedish", "EN": "English", "IT": "Italian", "ES": "Spanish", "NL": "Dutch", "RU": "Russian"}
KAIKKI = "https://kaikki.org/dictionary/{name}/kaikki.org-dictionary-{name}.jsonl"

# ---------------------------------------------------------------------------
section("1. OUR DEV WORDS (inputs only, no labels)")
dev = load_dataset(PROJECT / "work/data/dev")
targets = {}
for u in dev.usages:
    targets.setdefault((u.language, u.word), {"pos": u.pos, "definition": u.definition})
print(f"{'word':<11}{'POS in data':<13}target definition")
for (lang, w), t in sorted(targets.items()):
    print(f"{w:<11}{t['pos']:<13}{t['definition'][:70]}")

# ---------------------------------------------------------------------------
section("2. IS WIKTIONARY DATA AVAILABLE FOR ALL 6 TEST LANGUAGES?")
print("Source: kaikki.org (machine-readable English Wiktionary, free, CC BY-SA).")
print("Glosses are in English for every language, so the method is identical across languages.\n")
availability = {}
for code, name in LANG_NAME.items():
    url = KAIKKI.format(name=name)
    try:
        r = requests.head(url, allow_redirects=True, timeout=30)
        size = int(r.headers.get("Content-Length", 0)) / 2**20
        ok = r.status_code == 200
    except Exception as e:
        ok, size = False, 0
    availability[code] = {"url": url, "available": ok, "size_mb": round(size, 1)}
    print(f"  {name:<9} {'PASS' if ok else 'FAIL':<5} {size:>8.0f} MB   {url}")
all_ok = all(a["available"] for a in availability.values())
print(f"\nAll 6 languages available: {'PASS' if all_ok else 'FAIL: see the rows above'}")

# ---------------------------------------------------------------------------
section("3. DOWNLOAD SWEDISH AND EXTRACT OUR 10 WORDS")
url = availability["SV"]["url"]
print(f"Streaming {url}\n(only entries for our 10 words are kept)")
wanted = {w for (_, w) in targets}
entries, sha, n_lines, t0 = defaultdict(list), hashlib.sha256(), 0, time.time()
with requests.get(url, stream=True, timeout=120) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        sha.update(line + b"\n"); n_lines += 1
        if not line: continue
        obj = json.loads(line)
        if obj.get("word") in wanted:
            entries[obj["word"]].append(obj)
downloaded = datetime.now(timezone.utc).isoformat()
print(f"Read {n_lines:,} dictionary entries in {time.time() - t0:.0f}s")
print(f"File SHA-256 (records the exact Wiktionary version): {sha.hexdigest()[:16]}...")

# ---------------------------------------------------------------------------
section("4. DICTIONARY SENSES PER WORD")
print("Matched by part of speech, so e.g. 'skär' (adjective) does not pick up 'skär' (noun, skerry).")
print("Pure inflection entries ('form of ...') are skipped. Usage tags like 'dated' are kept.\n")
POS_MAP = {"NN": "noun", "NOUN": "noun", "VB": "verb", "VERB": "verb", "JJ": "adj", "ADJ": "adj",
           "noun": "noun", "verb": "verb", "adj": "adj", "adjective": "adj"}

def extract(word, data_pos):
    want = POS_MAP.get(data_pos)
    ents = entries.get(word, [])
    chosen = [e for e in ents if want and e.get("pos") == want] or ents   # fall back to all POS if no match
    senses = []
    for e in chosen:
        for s in e.get("senses", []):
            if "form_of" in s or "form-of" in s.get("tags", []):
                continue
            gl = s.get("glosses") or s.get("raw_glosses")
            if not gl: continue
            text = gl[-1].strip()
            tags = [t for t in s.get("tags", []) if t in ("dated", "archaic", "obsolete", "colloquial", "slang", "figuratively", "formal")]
            item = f"{text}" + (f" ({', '.join(tags)})" if tags else "")
            if item not in senses: senses.append(item)
    return senses, sorted({e.get("pos") for e in chosen}), sorted({e.get("pos") for e in ents})

results, low_coverage = {}, []
for (lang, w), t in sorted(targets.items()):
    senses, used_pos, all_pos = extract(w, t["pos"])
    results[f"{lang}|{w}"] = {"pos_in_data": t["pos"], "dictionary_pos_used": used_pos,
                              "senses": senses, "target_definition": t["definition"]}
    flag = "  <- LOW COVERAGE (fewer than 2 senses)" if len(senses) < 2 else ""
    if len(senses) < 2: low_coverage.append(w)
    print(f"--- {w.upper()}  (data POS: {t['pos']}; dictionary POS used: {used_pos}; available: {all_pos}){flag}")
    print(f"    Target: {t['definition'][:110]}")
    for i, s in enumerate(senses, 1):
        print(f"      {i}. {s[:130]}")
    if not senses:
        print("      (no senses found)")
    print()

save_json(RUN / "dictionary_senses_SV.json", {
    "source_url": url, "downloaded_utc": downloaded, "sha256": sha.hexdigest(),
    "availability_all_languages": availability, "words": results})
record_environment(PROJECT, RUN / "manifest.json", purpose="Wiktionary (kaikki.org) sense extraction, label-free",
                   source_url=url, source_sha256=sha.hexdigest())

# ---------------------------------------------------------------------------
section("SUMMARY")
n_ok = sum(len(r["senses"]) >= 2 for r in results.values())
print(f"Wiktionary available for all 6 test languages : {'PASS' if all_ok else 'FAIL'}")
print(f"Words with at least 2 dictionary senses       : {n_ok}/10")
print(f"Words with low coverage (method cannot help)  : {', '.join(low_coverage) or 'none'}")
print(f"Saved: {RUN / 'dictionary_senses_SV.json'}")
print("\nWhat to check before step 2 (A100):")
print("  1. Section 2: all 6 languages PASS, so the method can run on the test set.")
print("  2. Section 4: POS matching worked (e.g. SKÄR shows adjective senses, not 'skerry').")
print("  3. Section 4: DUMPA lists 'unload waste'. That is the sense behind its 79 false positives.")
print("No labels were used. No GPU used. Paste this output to continue.")

In [ ]:
import sys, subprocess, time, torch
def section(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

section("LOAD GEMMA 4 31B (same pinned version as our 0.678 result)")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "transformers", "accelerate", "safetensors", "huggingface_hub"], check=True)
from transformers import AutoProcessor, AutoModelForMultimodalLM
from semeval_task3.data import load_dataset

assert torch.cuda.is_available(), "Switch the runtime to A100."
print(f"GPU            : {torch.cuda.get_device_name(0)} ({torch.cuda.mem_get_info()[1] / 2**30:.0f} GiB)")

GEMMA_ID = "google/gemma-4-31B-it"
GEMMA_REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"
t0 = time.time()
gemma_processor = AutoProcessor.from_pretrained(GEMMA_ID, revision=GEMMA_REVISION)
gemma_model = AutoModelForMultimodalLM.from_pretrained(
    GEMMA_ID, revision=GEMMA_REVISION, dtype=torch.bfloat16,
    device_map={"": 0}, attn_implementation="sdpa").eval()
gemma_model.requires_grad_(False)
gemma_tokenizer = gemma_processor.tokenizer
print(f"Model loaded   : PASS ({time.time() - t0:.0f}s, {torch.cuda.memory_allocated() / 2**30:.1f} GiB on GPU)")

yes_ids = gemma_tokenizer.encode("Yes", add_special_tokens=False)
no_ids = gemma_tokenizer.encode("No", add_special_tokens=False)
assert len(yes_ids) == len(no_ids) == 1
GEMMA_YES_ID, GEMMA_NO_ID = yes_ids[0], no_ids[0]
print(f"Yes/No tokens  : {'PASS' if (GEMMA_YES_ID, GEMMA_NO_ID) == (10784, 3771) else 'CHECK'} "
      f"(Yes={GEMMA_YES_ID}, No={GEMMA_NO_ID}; expected 10784 / 3771)")

gemma_dev = load_dataset(PROJECT / "work/data/dev")   # inputs only, no labels
assert not any(u.span_flags for u in gemma_dev.usages)
print(f"Dev sentences  : {len(gemma_dev.usages)} loaded (no labels)")
print("\nGemma is ready. Run the scoring cell next.")

In [ ]:
import json, time, numpy as np, torch
from sklearn.metrics import roc_auc_score
from project_utils import new_run, save_json, save_scores, read_scores, frozen, save_evaluation, record_environment
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap

def section(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

LANG_NAME = {"SV": "Swedish", "EN": "English", "IT": "Italian", "ES": "Spanish", "NL": "Dutch", "RU": "Russian"}
V1_SCORES = PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json"
V1_ZIP = PROJECT / "outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a/gemma_raw.zip"

# ---------------------------------------------------------------------------
section("1. LOAD DICTIONARY SENSES (from step 1, unedited)")
dict_files = sorted((PROJECT / "outputs/runs/step06_dictionary_senses").glob("*/dictionary_senses_SV.json"))
assert dict_files, "Run the step-1 notebook first."
DICT = json.loads(dict_files[-1].read_text())["words"]
print(f"Source file: {dict_files[-1]}")
for key, v in sorted(DICT.items()):
    print(f"  {key.split('|')[1]:<11} {len(v['senses'])} dictionary senses")

# ---------------------------------------------------------------------------
section("2. THE PROMPT (Gemma v1 + dictionary block; final question unchanged)")
def dict_messages(u):
    lang = LANG_NAME[u.language]
    senses = DICT.get(f"{u.language}|{u.word}", {}).get("senses", [])
    block = ""
    if senses:
        block = ("For reference, a dictionary lists these senses of this word "
                 "(one of them may correspond to the target sense):\n"
                 + "\n".join(f"{i}. {s}" for i, s in enumerate(senses, 1)) + "\n")
    return [{"role": "system", "content": f"You are an expert lexicographer of historical {lang}."},
            {"role": "user", "content": (
                f"Target word (lemma): {u.word}\n"
                f"Sense definition ({lang}): {u.definition}\n"
                f"{block}"
                f"Sentence from {u.year}: {u.marked_text()}\n\n"
                "Does the word marked with <t> and </t> in the sentence express "
                "the sense given by the definition? Answer only Yes or No.")}]
example = next(u for u in gemma_dev.usages if u.word == "dumpa")
print(dict_messages(example)[1]["content"])

# ---------------------------------------------------------------------------
section("3. SCORING ALL 1,983 USAGES WITH GEMMA")
RUN = new_run(PROJECT, "step06_dictionary_senses_scoring")
gemma_tokenizer.padding_side = "left"
scores, low_mass, t0 = {}, 0, time.time()
with torch.inference_mode():
    for i in range(0, len(gemma_dev.usages), 4):
        batch = gemma_dev.usages[i:i + 4]
        inputs = gemma_processor.apply_chat_template(
            [dict_messages(u) for u in batch], tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, enable_thinking=False,
            processor_kwargs={"padding": True, "truncation": False}).to("cuda")
        assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
        logits = gemma_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
        p = torch.softmax(logits, -1)
        low_mass += int(((p[:, GEMMA_YES_ID] + p[:, GEMMA_NO_ID]) < 0.9).sum())
        scores.update({u.key: v for u, v in zip(batch, (logits[:, GEMMA_YES_ID] - logits[:, GEMMA_NO_ID]).tolist())})
        if i % 400 == 0:
            print(f"  {i:>5}/{len(gemma_dev.usages)} scored  ({time.time() - t0:.0f}s)")
save_scores(RUN / "gemma_dictionary_scores.json", scores)
print(f"\nScored               : {len(scores)} usages in {time.time() - t0:.0f}s")
print(f"Yes/No answer health : {'PASS' if low_mass == 0 else f'WARNING: {low_mass} answers below 90% Yes/No mass'}")
print(f"Score range          : {min(scores.values()):.2f} to {max(scores.values()):.2f}")

# ---------------------------------------------------------------------------
section("4. HELD-OUT EVALUATION (same nested CV, same grid as Gemma v1)")
eval_dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = eval_dev.require_gold()
grid = tuple([-1000.0] + [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)] + [1000.0])
assert all(-40 <= v <= 40 for v in scores.values()), "Scores outside grid."
res = nested_word_cv(eval_dev, [Candidate("gemma_dictionary", frozen(scores), grid)])
save_evaluation(RUN, "gemma_dictionary", res, eval_dev, PROJECT, official=True)
print("Official scorer agreement: PASS")

v1 = score(gold, read_submission(V1_ZIP, eval_dev.keys))
assert abs(v1["average"] - 0.6782392862783919) < 1e-10, "Gemma v1 reference did not reproduce."
v1_scores = read_scores(V1_SCORES, eval_dev)
pair = word_bootstrap(res["report"], other=v1)
v1w = {r["word"]: r for r in v1["per_word"]}

# ---------------------------------------------------------------------------
section("5. PER-WORD COMPARISON: Gemma v1 vs Gemma + dictionary")
print("F1 = competition metric (uses the chosen threshold).")
print("AUC = how well the scores rank Yes above No, regardless of threshold.\n")
print(f"{'word':<11}{'senses':>7}{'v1 F1':>8}{'dict F1':>9}{'change':>9}{'v1 AUC':>8}{'dict AUC':>10}")
wins, a1s, a2s = 0, [], []
for r in res["report"]["per_word"]:
    w = r["word"]; ks = [u.key for u in eval_dev.usages if u.word == w]; y = [gold[k] for k in ks]
    a1, a2 = roc_auc_score(y, [v1_scores[k] for k in ks]), roc_auc_score(y, [scores[k] for k in ks])
    a1s.append(a1); a2s.append(a2)
    d = r["f1"] - v1w[w]["f1"]; wins += d > 0
    n = len(DICT.get(f"SV|{w}", {}).get("senses", []))
    print(f"{w:<11}{n:>7}{v1w[w]['f1']:>8.3f}{r['f1']:>9.3f}{d:>+9.3f}{a1:>8.3f}{a2:>10.3f}")
print(f"{'MEAN':<11}{'':>7}{v1['average']:>8.3f}{res['report']['average']:>9.3f}"
      f"{res['report']['average'] - v1['average']:>+9.3f}{np.mean(a1s):>8.3f}{np.mean(a2s):>10.3f}")
print("\nThresholds chosen per held-out word:", [float(s["threshold"]) for s in res["selections"]])

# ---------------------------------------------------------------------------
adopt = pair["estimate"] >= 0.02 and wins >= 6
section("DECISION")
print(f"Gemma + dictionary minus Gemma v1 : {pair['estimate']:+.4f}  (95% interval {pair['lower']:+.4f} to {pair['upper']:+.4f})")
print(f"Words improved                    : {wins}/10")
print(f"Rule (set in advance)             : adopt only if gain >= +0.02 AND improved on >= 6/10 words")
print(f"\nDECISION: {'ADOPT Gemma + dictionary' if adopt else 'KEEP Gemma v1'}")
save_json(RUN / "decision.json", {"rule": "adopt if paired gain >= +0.02 and better on >= 6/10 words",
                                   "paired_vs_v1": pair, "words_better": wins, "adopt": adopt,
                                   "mean_auc_v1": float(np.mean(a1s)), "mean_auc_dictionary": float(np.mean(a2s)),
                                   "dictionary_file": str(dict_files[-1])})
record_environment(PROJECT, RUN / "manifest.json", model_revision=GEMMA_REVISION,
                   prompt="Gemma v1 + Wiktionary sense list", dictionary_file=str(dict_files[-1]))
print(f"\nSaved: {RUN}")
print("Disconnect the A100 after copying this output.")

In [ ]:
# THROWAWAY MOCK TEST: 6 languages from Wiktionary quotations. Delete this cell after copying the output.
import json, re, random, time, requests, numpy as np, torch
from collections import defaultdict
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from semeval_task3.data import Usage
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)

THRESHOLD = 10.5
LANG_NAME = {"SV": "Swedish", "EN": "English", "IT": "Italian", "ES": "Spanish", "NL": "Dutch", "RU": "Russian"}
KAIKKI = "https://kaikki.org/dictionary/{name}/kaikki.org-dictionary-{name}.jsonl"
POS_OK, WORDS_PER_LANG = {"noun", "verb", "adj"}, 100
rng = random.Random(2027)
ACCENTS = dict.fromkeys(map(ord, "\u0301\u0300"), None)   # strip Russian stress marks
clean = lambda t: t.translate(ACCENTS)

# Final prompt (Gemma v1 with language templated). Mock meanings are English, so they are labelled English.
def final_messages(u):
    lang = LANG_NAME[u.language]
    return [{"role": "system", "content": f"You are an expert lexicographer of historical {lang}."},
            {"role": "user", "content": (
                f"Target word (lemma): {u.word}\n"
                f"Sense definition (English): {u.definition}\n"
                f"Sentence from {u.year}: {u.marked_text()}\n\n"
                "Does the word marked with <t> and </t> in the sentence express "
                "the sense given by the definition? Answer only Yes or No.")}]

def year_of(ref):
    m = re.findall(r"\b(1[5-9]\d\d|20[0-2]\d)\b", ref or "")
    return int(m[0]) if m else None

def find_target(text, word, ex):
    ctext = clean(text)
    offs = ex.get("bold_text_offsets") or []
    if offs:
        surf = clean(text[offs[0][0]:offs[0][1]]).strip()
        if surf and ctext.count(surf) == 1:
            i = ctext.index(surf); return ctext, i, i + len(surf)
    w = clean(word).lower(); stem = w[:max(3, len(w) - 2)]
    hits = [m for m in re.finditer(r"\w+", ctext) if m.group().lower().startswith(stem)]
    return (ctext, hits[0].start(), hits[0].end()) if len(hits) == 1 else None

# ---------------------------------------------------------------------------
section("1. BUILDING THE MOCK TEST FROM WIKTIONARY")
print("Each question = a dated quotation + one meaning of the target word.")
print("Correct answer YES for the quotation's own meaning, NO for another meaning of the same word.\n")
usages, labels = [], {}
for code, name in LANG_NAME.items():
    t0, cands = time.time(), defaultdict(list)
    with requests.get(KAIKKI.format(name=name), stream=True, timeout=300) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line: continue
            e = json.loads(line); w = e.get("word", "")
            if e.get("pos") not in POS_OK or " " in w or len(w) < 3: continue
            for s in e.get("senses", []):
                if "form_of" in s or "form-of" in s.get("tags", []): continue
                gl = (s.get("glosses") or [""])[-1].strip()
                if len(gl) < 4: continue
                for ex in s.get("examples", []):
                    y, txt = year_of(ex.get("ref")), ex.get("text", "")
                    if not y or len(txt.split()) < 5 or len(txt) > 600: continue
                    tgt = find_target(txt, w, ex)
                    if tgt:
                        cands[(w, e["pos"])].append((gl, tgt, y)); break
    eligible = [k for k, v in cands.items() if len({g for g, _, _ in v}) >= 2]
    rng.shuffle(eligible)
    n_before, yrs = len(usages), []
    for (w, pos) in eligible[:WORDS_PER_LANG]:
        senses = {}
        for g, tgt, y in cands[(w, pos)]:
            senses.setdefault(g, (tgt, y))
        (g1, a1), (g2, a2) = rng.sample(sorted(senses.items()), 2)
        for g_own, (tgt, y), g_other in [(g1, a1, g2), (g2, a2, g1)]:
            text, s0, s1 = tgt
            for label, gloss in [(1, g_own), (0, g_other)]:
                u = Usage(code, w, f"mock{len(usages):05d}", text, gloss, s0, s1, y, "mock", pos)
                usages.append(u); labels[u.key] = label; yrs.append(y)
    n_words = min(len(eligible), WORDS_PER_LANG)
    flag = "  <- TOO FEW, unreliable" if n_words < 30 else ""
    print(f"{name:<9} {time.time() - t0:>4.0f}s | eligible words {len(eligible):>6} | used {n_words:>3} -> "
          f"{len(usages) - n_before:>3} questions | years {min(yrs) if yrs else '-'} to {max(yrs) if yrs else '-'} "
          f"(median {int(np.median(yrs)) if yrs else '-'}){flag}")

ex = next((u for u in usages if u.language == "RU"), usages[0])
print(f"\nExample question ({LANG_NAME[ex.language]}):")
print(final_messages(ex)[1]["content"])
print(f"Correct answer: {'YES' if labels[ex.key] else 'NO'}")

# ---------------------------------------------------------------------------
section("2. SCORING WITH GEMMA (final prompt, final cutoff)")
gemma_tokenizer.padding_side = "left"
scores, low, t0 = {}, 0, time.time()
with torch.inference_mode():
    for i in range(0, len(usages), 4):
        batch = usages[i:i + 4]
        inputs = gemma_processor.apply_chat_template(
            [final_messages(u) for u in batch], tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, enable_thinking=False,
            processor_kwargs={"padding": True, "truncation": False}).to("cuda")
        assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
        logits = gemma_model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
        p = torch.softmax(logits, -1)
        low += int(((p[:, GEMMA_YES_ID] + p[:, GEMMA_NO_ID]) < 0.9).sum())
        scores.update({u.key: v for u, v in zip(batch, (logits[:, GEMMA_YES_ID] - logits[:, GEMMA_NO_ID]).tolist())})
        if i % 400 == 0:
            print(f"  {i:>5}/{len(usages)} scored ({time.time() - t0:.0f}s)")
print(f"Done in {time.time() - t0:.0f}s | Yes/No answer health: {'PASS' if low == 0 else f'WARNING: {low} low-mass answers'}")

# ---------------------------------------------------------------------------
section("3. RESULTS PER LANGUAGE")
print("F1 @10.5 = F1 with our frozen cutoff (what we will do in January)")
print("Best t   = best possible cutoff for that language (needs labels; reference only)")
print("Cost     = F1 lost by using 10.5 instead of the best cutoff (want < 0.02)")
print("AUC      = how well Gemma separates right from wrong meanings (1.0 = perfect)")
print("Medians  = typical score for correct-YES and correct-NO questions (similar across languages = stable scale)\n")
grid = [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)]
f1_at = lambda y, s, t: precision_recall_fscore_support(y, (s >= t).astype(int), average="binary", zero_division=0)[2]
print(f"{'language':<9}{'Qs':>5}{'F1@10.5':>9}{'best t':>8}{'best F1':>9}{'cost':>8}{'AUC':>7}{'med YES':>9}{'med NO':>8}   verdict")
summary = {}
for code, name in LANG_NAME.items():
    keys = [u.key for u in usages if u.language == code]
    if len(keys) < 60:
        print(f"{name:<9}{len(keys):>5}   too few questions to judge"); continue
    y = np.array([labels[k] for k in keys]); s = np.array([scores[k] for k in keys])
    f_frozen = f1_at(y, s, THRESHOLD)
    t_best = max(grid, key=lambda t: (f1_at(y, s, t), -t)); f_best = f1_at(y, s, t_best)
    cost, auc = f_best - f_frozen, roc_auc_score(y, s)
    verdict = "OK" if cost < 0.02 else ("WATCH" if cost < 0.05 else "PROBLEM: cutoff does not transfer")
    summary[code] = dict(questions=len(keys), f1_at_10_5=f_frozen, best_threshold=t_best, f1_best=f_best,
                         cost=cost, auc=auc, median_yes=float(np.median(s[y == 1])), median_no=float(np.median(s[y == 0])))
    print(f"{name:<9}{len(keys):>5}{f_frozen:>9.3f}{t_best:>8}{f_best:>9.3f}{cost:>8.3f}{auc:>7.3f}"
          f"{summary[code]['median_yes']:>9.1f}{summary[code]['median_no']:>8.1f}   {verdict}")

MOCK = new_run(PROJECT, "mock_test_6_languages")
save_json(MOCK / "mock_results.json", summary)
section("HOW TO READ THIS")
print("- Compare languages with EACH OTHER, not with our Swedish 0.678: these questions are easier")
print("  (dictionary quotations, 50% YES), so every F1 here will be higher than the real task.")
print("- 'cost' is the key column: under 0.02 everywhere means the 10.5 cutoff transfers.")
print("- A language with clearly lower AUC than the others is one where Gemma itself is weaker.")
print(f"\nResults saved (small file only): {MOCK / 'mock_results.json'}")
print("You can now delete this cell. Disconnect the A100 after copying the output.")